In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

In [4]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-13 19:00:38--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

     0K ..                                                    100% 18.2M=0s

2026-07-13 19:00:38 (18.2 MB/s) - ‘rag_helper.py’ saved [2134/2134]

--2026-07-13 19:00:38--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py’

     0K ...                                         

In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from evaluation_utils import llm_structured

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
target_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

target_docs = [doc for doc in documents if doc["filename"] in target_filenames]
len(target_docs), [doc["filename"] for doc in target_docs]

(3,
 ['01-agentic-rag/lessons/01-intro.md',
  '01-agentic-rag/lessons/02-environment.md',
  '01-agentic-rag/lessons/03-rag.md'])

In [12]:
import json

def generate_questions_for_doc(doc):
    user_prompt = json.dumps(
        {
            "filename": doc["filename"],
            "content": doc["content"],
        },
        ensure_ascii=False,
    )

    parsed, usage = llm_structured(
        client=openai_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="gpt-5.4-mini",
    )

    records = [
        {
            "question": question,
            "filename": doc["filename"],
        }
        for question in parsed.questions
    ]

    return {
        "filename": doc["filename"],
        "questions": parsed.questions,
        "records": records,
        "usage": usage,
    }

In [13]:
results = [generate_questions_for_doc(doc) for doc in target_docs]
len(results)

3

In [14]:
results

[{'filename': '01-agentic-rag/lessons/01-intro.md',
  'questions': ['Why is RAG useful instead of just asking the model directly?',
   'What does this course build in the first part of the module?',
   'What are the main limits of large language models mentioned here?',
   'How does this lesson suggest we learn the RAG system before using a framework?',
   'What changes in the second part when the pipeline becomes agentic?'],
  'records': [{'question': 'Why is RAG useful instead of just asking the model directly?',
    'filename': '01-agentic-rag/lessons/01-intro.md'},
   {'question': 'What does this course build in the first part of the module?',
    'filename': '01-agentic-rag/lessons/01-intro.md'},
   {'question': 'What are the main limits of large language models mentioned here?',
    'filename': '01-agentic-rag/lessons/01-intro.md'},
   {'question': 'How does this lesson suggest we learn the RAG system before using a framework?',
    'filename': '01-agentic-rag/lessons/01-intro.md

In [15]:
for r in results:
    print(r["filename"])
    print("input_tokens:", r["usage"].input_tokens)
    print("output_tokens:", r["usage"].output_tokens)
    print("questions:")
    for q in r["questions"]:
        print("-", q)
    print()

01-agentic-rag/lessons/01-intro.md
input_tokens: 1016
output_tokens: 83
questions:
- Why is RAG useful instead of just asking the model directly?
- What does this course build in the first part of the module?
- What are the main limits of large language models mentioned here?
- How does this lesson suggest we learn the RAG system before using a framework?
- What changes in the second part when the pipeline becomes agentic?

01-agentic-rag/lessons/02-environment.md
input_tokens: 1282
output_tokens: 120
questions:
- What do I need installed before starting this module, and is anything besides Python and Jupyter required?
- How do I create the project from scratch and which packages should I add first?
- What’s the safest way to keep my API key out of git when I’m working on this course?
- How do I launch Jupyter and test that the OpenAI client is set up correctly in a notebook?
- If I want to use Groq or another OpenAI-style provider, what changes do I need to make to the client setup?



In [16]:
question_records = []
input_tokens = []

for r in results:
    question_records.extend(r["records"])
    input_tokens.append(r["usage"].input_tokens)

avg_input_tokens = sum(input_tokens) / len(input_tokens)

avg_input_tokens, input_tokens, len(question_records)

(1349.0, [1016, 1282, 1749], 15)

In [17]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-13 19:18:06--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

     0K .......... .......... .......... .......... .......   100% 1.91M=0.02s

2026-07-13 19:18:07 (1.91 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



In [18]:
import pandas as pd

In [19]:
df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [20]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [23]:
len(chunks), chunks[0].keys()

(295, dict_keys(['start', 'content', 'filename']))

In [24]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [25]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [26]:
text_search("What is RAG?", num_results=2)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [36]:
import numpy as np
from embedder import Embedder

In [34]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [39]:
embedder = Embedder()
X = embedder.encode_batch([chunk["content"] for chunk in chunks])

In [40]:
X.shape

(295, 384)

In [43]:
def vector_search(query, num_results=5):
    v = embedder.encode(query)
    scores = X.dot(v)
    top_idx = np.argsort(scores)[::-1][:num_results]
    return [chunks[i] for i in top_idx]

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [44]:
q = ground_truth[0]["question"]

print(q)
print(text_search(q, 1)[0]["filename"])
print(vector_search(q, 1)[0]["filename"])
print(hybrid_search(q, 60)[0]["filename"])

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/01-intro.md


In [45]:
q = ground_truth[0]["question"]
q
text_results = text_search(q, num_results=5)
text_results[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [46]:
vector_results = vector_search(q, num_results=5)
vector_results[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [48]:
hybrid_results = hybrid_search(q)
hybrid_results[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [49]:
from tqdm.auto import tqdm

In [50]:
def compute_relevance(rec, search_function):
    question = rec["question"]
    expected_filename = rec["filename"]

    results = search_function(question)

    relevance = []
    for doc in results:
        relevance.append(doc["filename"] == expected_filename)

    return relevance

In [51]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

In [52]:
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank, rel in enumerate(line):
            if rel:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

In [53]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for rec in tqdm(ground_truth):
        relevance = compute_relevance(rec, search_function)
        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [54]:
evaluate(ground_truth, text_search)


  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [55]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [56]:
for k in [1, 50, 100, 200]:
    metrics = evaluate(ground_truth, lambda q, k=k: hybrid_search(q, k=k))
    print(k, metrics)

  0%|          | 0/360 [00:00<?, ?it/s]

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
